# Study 1: Within-Cohort Cross-Validation

ML autism prediction with **within-cohort** train/test and CV on three cohorts:
- **C4**: Processed in `data_pipeline_recreation.ipynb`
- **CARD**: Processed in `card_c4_validation.ipynb` (harmonized to C4 schema)
- **Dataset3 (YBT)**: EQ-10, SQ-R-10, AQ-10, demographics (no SPQ)

Methodology:
1. Age restriction 18+ for all cohorts (no upper cutoff)
2. AQ >= 6 filter for autism cases only
3. Balance 50/50 autism / non-autism
4. Stratified 80/20 train/test
5. 5-fold CV on training set; evaluate on held-out test
6. Models: XGBoost, LightGBM, Random Forest, Logistic Regression
7. Metrics: AUROC, Sensitivity, Specificity, F1, PPV, NPV

In [ ]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd

_cwd = os.path.abspath(os.getcwd())
REPO_ROOT = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'notebooks' else _cwd
if not os.path.isdir(os.path.join(REPO_ROOT, 'data')):
    REPO_ROOT = os.path.dirname(REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))

from study_utils import (
    load_cohort_c4,
    load_cohort_card,
    load_cohort_ybt,
    create_stratified_split,
    train_with_cv,
    evaluate_model,
    get_models,
    RESULTS_DIR,
    FEATURE_NAMES_45,
    FEATURE_NAMES_35,
)

C4_PATH = os.path.join(REPO_ROOT, 'data', 'processed', 'data_c4_final_recreated_cleaned.csv')
CARD_ALIGNED_PATH = os.path.join(REPO_ROOT, 'data', 'processed', 'card_aligned.csv')
YBT_ALIGNED_PATH = os.path.join(REPO_ROOT, 'data', 'processed', 'ybt_aligned.csv')
_default_ybt = os.path.expanduser('~/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/YBT.csv')
_repo_ybt = os.path.join(REPO_ROOT, 'data', 'raw', 'YBT.csv')
YBT_RAW_PATH = os.environ.get('YBT_PATH', _default_ybt if os.path.isfile(_default_ybt) else _repo_ybt)

STUDY1_DIR = os.path.join(RESULTS_DIR, 'study1_within_cohort')
os.makedirs(STUDY1_DIR, exist_ok=True)
print('REPO_ROOT:', REPO_ROOT)
print('C4 (data_pipeline_recreation output):', os.path.isfile(C4_PATH))
print('CARD (card_c4_validation output):', os.path.isfile(CARD_ALIGNED_PATH))
print('YBT aligned (external_validation_ybt output):', os.path.isfile(YBT_ALIGNED_PATH))
print('YBT raw:', os.path.isfile(YBT_RAW_PATH))

## 1. Load and preprocess C4

In [ ]:
df_c4, feat_c4, target_c4 = load_cohort_c4(C4_PATH, age_min=18, age_max=120, balance_50_50=True, apply_aq_filter=True)
print('C4 shape:', df_c4.shape)
print('C4 diagnosis counts:', df_c4[target_c4].value_counts().to_dict())
print('Features:', len(feat_c4))

## 2. Load CARD (preprocessed 45-feature data)

Uses `data/processed/card_aligned.csv` produced by **card_c4_validation.ipynb** (same CARD_PATH as in that notebook). Run that notebook first to generate the file.

In [ ]:
if os.path.isfile(CARD_ALIGNED_PATH):
    df_card, feat_card, target_card = load_cohort_card(CARD_ALIGNED_PATH, age_min=18, age_max=120, balance_50_50=True, apply_aq_filter=True)
    print('CARD shape:', df_card.shape)
    print('CARD diagnosis counts:', df_card[target_card].value_counts().to_dict())
else:
    df_card, feat_card, target_card = None, None, None
    print('CARD aligned file not found at', CARD_ALIGNED_PATH)
    print('Run card_c4_validation.ipynb (it saves card_aligned.csv to data/processed/)')

## 3. Load Dataset3 (YBT)

In [ ]:
ybt_path = YBT_ALIGNED_PATH if os.path.isfile(YBT_ALIGNED_PATH) else YBT_RAW_PATH
df_ybt, feat_ybt, target_ybt = load_cohort_ybt(ybt_path, age_min=18, age_max=120, balance_50_50=True, apply_aq_filter=True)
print('YBT shape:', df_ybt.shape)
print('YBT diagnosis counts:', df_ybt[target_ybt].value_counts().to_dict())
print('Features:', len(feat_ybt), '(45 from aligned, 35 from raw)' if len(feat_ybt) == 45 else '(35 from raw)')

## 4. Train and evaluate per cohort

For each cohort: stratified split, train 4 models with 5-fold CV, evaluate on test, save models and metrics.

In [ ]:
def run_cohort(cohort_name, df, feature_names, target_col, has_spq=True):
    if df is None or len(df) < 100:
        print(f'Skipping {cohort_name}: no data')
        return
    X = df[feature_names].fillna(0).values
    y = df[target_col].values
    train_df, test_df = create_stratified_split(df, target_col=target_col)
    X_train = train_df[feature_names].fillna(0).values
    y_train = train_df[target_col].values
    X_test = test_df[feature_names].fillna(0).values
    y_test = test_df[target_col].values

    out_dir = os.path.join(STUDY1_DIR, cohort_name)
    os.makedirs(os.path.join(out_dir, 'models'), exist_ok=True)

    scaler = None
    all_metrics = {}
    cv_scores = {}
    importance_list = []

    for model_name, model in get_models().items():
        fitted, scaler, cv_metrics, opt_thresh = train_with_cv(model, X_train, y_train, scaler=scaler, fit_scaler=(scaler is None))
        X_test_s = scaler.transform(X_test)
        test_metrics = evaluate_model(fitted, X_test_s, y_test, threshold=opt_thresh)
        all_metrics[model_name] = {
            'test_auroc': test_metrics['auroc'], 'test_f1': test_metrics['f1'],
            'test_sensitivity': test_metrics['sensitivity'], 'test_specificity': test_metrics['specificity'],
            'test_ppv': test_metrics['ppv'], 'test_npv': test_metrics['npv'],
            'cv_auroc_mean': cv_metrics['cv_auroc_mean'], 'cv_auroc_std': cv_metrics['cv_auroc_std'],
            'optimal_threshold': opt_thresh,
        }
        cv_scores[model_name] = cv_metrics
        joblib.dump(fitted, os.path.join(out_dir, 'models', f'{model_name}_model.joblib'))
        if hasattr(fitted, 'feature_importances_'):
            for i, imp in enumerate(fitted.feature_importances_):
                importance_list.append({'model': model_name, 'feature': feature_names[i], 'importance': float(imp)})

    joblib.dump(scaler, os.path.join(out_dir, 'scaler.joblib'))
    with open(os.path.join(out_dir, 'performance_metrics.json'), 'w') as f:
        json.dump(all_metrics, f, indent=2)
    with open(os.path.join(out_dir, 'cv_scores.json'), 'w') as f:
        json.dump(cv_scores, f, indent=2)
    if importance_list:
        pd.DataFrame(importance_list).to_csv(os.path.join(out_dir, 'feature_importance.csv'), index=False)
    print(cohort_name, 'done. Test AUROC (XGB):', all_metrics.get('xgboost', {}).get('test_auroc'))
    return all_metrics

In [ ]:
metrics_c4 = run_cohort('c4', df_c4, feat_c4, target_c4, has_spq=True)
if df_card is not None:
    metrics_card = run_cohort('card', df_card, feat_card, target_card, has_spq=True)
metrics_ybt = run_cohort('dataset3', df_ybt, feat_ybt, target_ybt, has_spq=False)

## 5. Comparison table across cohorts

In [ ]:
rows = []
for cohort, name in [('c4', 'C4'), ('card', 'CARD'), ('dataset3', 'Dataset3')]:
    p = os.path.join(STUDY1_DIR, cohort, 'performance_metrics.json')
    if os.path.isfile(p):
        with open(p) as f:
            m = json.load(f)
        for model_name, v in m.items():
            rows.append({'Cohort': name, 'Model': model_name, 'Test_AUROC': v['test_auroc'], 'Test_F1': v['test_f1'], 'Test_Sens': v['test_sensitivity'], 'Test_Spec': v['test_specificity']})
if rows:
    summary = pd.DataFrame(rows)
    print(summary.to_string(index=False))
    summary.to_csv(os.path.join(STUDY1_DIR, 'comparison_table.csv'), index=False)

## 6. ROC curves (optional)

Plot ROC curves per cohort using the best model (e.g. XGBoost).

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (cohort, title) in zip(axes, [('c4', 'C4'), ('card', 'CARD'), ('dataset3', 'Dataset3')]):
    p = os.path.join(STUDY1_DIR, cohort, 'performance_metrics.json')
    if not os.path.isfile(p):
        ax.set_title(title + ' (no data)')
        continue
    model_path = os.path.join(STUDY1_DIR, cohort, 'models', 'xgboost_model.joblib')
    scaler_path = os.path.join(STUDY1_DIR, cohort, 'scaler.joblib')
    if os.path.isfile(model_path) and os.path.isfile(scaler_path):
        # Reload test set from cohort run would require saving it; skip or re-split
        ax.plot([0, 1], [0, 1], 'k--')
        ax.set_xlabel('FPR')
        ax.set_ylabel('TPR')
        ax.set_title(title)
    else:
        ax.set_title(title)
plt.tight_layout()
plt.savefig(os.path.join(STUDY1_DIR, 'roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()